In [58]:
from dotenv import load_dotenv 
import os

import httpx
from time import sleep
import json
from pprint import pprint
from datetime import date, datetime

from dataclasses import dataclass, field, asdict
import random

load_dotenv()
api_key = os.getenv("FRED_API_KEY")

## Getting a basic call working

In [2]:
params = {
    "series_id": "UNRATE",
    "api_key": api_key,
    "file_type": "json",
    "observation_start": "2015-01-01",
}

url = "https://api.stlouisfed.org/fred/series/observations"

response = httpx.get(url, params=params)

In [3]:
data = json.loads(response.text)
observations = data["observations"]

In [6]:
observations

[{'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-01-01',
  'value': '5.7'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-02-01',
  'value': '5.5'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-03-01',
  'value': '5.4'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-04-01',
  'value': '5.4'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-05-01',
  'value': '5.6'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-06-01',
  'value': '5.3'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-07-01',
  'value': '5.2'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-08-01',
  'value': '5.1'},
 {'realtime_start': '2026-07-02',
  'realtime_end': '2026-07-02',
  'date': '2015-09-01',
  'value': '5.0'},
 {'realtime_start':

In [4]:
observations[0].get("date")

'2015-01-01'

In [5]:
print("First three observations:", observations[:3])
print("Last three observations:", observations[-3:])

First three observations: [{'realtime_start': '2026-07-02', 'realtime_end': '2026-07-02', 'date': '2015-01-01', 'value': '5.7'}, {'realtime_start': '2026-07-02', 'realtime_end': '2026-07-02', 'date': '2015-02-01', 'value': '5.5'}, {'realtime_start': '2026-07-02', 'realtime_end': '2026-07-02', 'date': '2015-03-01', 'value': '5.4'}]
Last three observations: [{'realtime_start': '2026-07-02', 'realtime_end': '2026-07-02', 'date': '2026-04-01', 'value': '4.3'}, {'realtime_start': '2026-07-02', 'realtime_end': '2026-07-02', 'date': '2026-05-01', 'value': '4.3'}, {'realtime_start': '2026-07-02', 'realtime_end': '2026-07-02', 'date': '2026-06-01', 'value': '4.2'}]


## Testing the dataset building logic

In [ ]:
@dataclass
class ObservationEntry:
    """Class for a question with a golden response from an API response."""
    input: str = field(init=False)
    target: float
    series_id: str
    series_name: str
    obs_date: date | str
    period_start: int
    period_end: int
    tolerance: float

    def __post_init__(self):
        # Define the input field value based on results
        obs_date_str = self.obs_date.strftime("%B %Y")
        self.input = f"What was the {self.series_name} in the United States in {obs_date_str}? Answer with only the number - nothing else in the response."

        # Update obs_date to string for JSON parsing
        self.obs_date = str(self.obs_date)

In [39]:
# Generate lists to iterate through / use for randomization
SERIES = {
    "UNRATE": "Unemployment Rate"
}

SERIES_FULL = {
    "UNRATE": "Unemployment Rate",
    "CIVPART": "Labor Force Participation Rate",
    "FEDFUNDS": "Effective Federal Funds Rate",
    "CPIAUCSL": "Consumer Price Index: All Urban Consumers, All Items",
    "CPILFESL": "CPI: All Items Less Food and Energy",
    "PAYEMS":   "Total Nonfarm Payroll Employment",
    "HOUST":    "New Privately-Owned Housing Units Started",
    "INDPRO":   "Industrial Production Index"
}

TOLERANCE = {
    "UNRATE": 0.1,
    "CIVPART": 0.1,
    "FEDFUNDS": 0.05,
    "CPIAUCSL": 0.5,
    "CPILFESL": 0.5,
    "PAYEMS": 200,
    "HOUST": 30,
    "INDPRO": 0.5
}

YEARS = [(2015, 2019), (2020, 2024), (2025, 2025)]
MONTHS = [m for m in range(1, 13)]

In [75]:
# Testing random building logic

series_id = "UNRATE"
series_name = SERIES["UNRATE"]
tolerance = TOLERANCE["UNRATE"]
random.seed(42)

obs_list = []

# Loop through each year range
for period_start, period_end in YEARS:
    print(f"Evaluating series: {series_id} for {period_start} - {period_end}")

    # Pick three random dates in that range, skipping any duplicates
    period_dates = []
    while len(period_dates) < 3:
        year = random.randint(period_start, period_end)
        month = random.choice(MONTHS)
        selected = date(year, month, 1)

        # Add selected dates to list if not already present
        if selected not in period_dates:
            period_dates.append(selected)
        else:
            print("\t\tDuplicate - skipping", selected)


    for obs in observations:
        # Convert observation date to a datetime date
        obs_date = datetime.strptime(obs.get("date"), "%Y-%m-%d").date()
        if obs_date in period_dates:
            print(f"Date match: {obs_date}")
            target = obs.get("value")

            obs_list.append(ObservationEntry(
                target = target,
                series_id = series_id,
                series_name = series_name,
                obs_date = obs_date,
                period_start = period_start,
                period_end = period_end,
                tolerance = tolerance
                )
            )

obs_list

Evaluating series: UNRATE for 2015 - 2019
Date match: 2015-01-01
Date match: 2016-03-01
Date match: 2017-04-01
Evaluating series: UNRATE for 2020 - 2024
Date match: 2020-11-01
Date match: 2024-02-01
Date match: 2024-07-01
Evaluating series: UNRATE for 2025 - 2025
Date match: 2025-01-01
Date match: 2025-04-01
Date match: 2025-09-01


[ObservationEntry(input='What was the Unemployment Rate in the United States in January 2015? Answer with only the number - nothing else in the response.', target='5.7', series_id='UNRATE', series_name='Unemployment Rate', obs_date='2015-01-01', period_start=2015, period_end=2019, tolerance=0.1),
 ObservationEntry(input='What was the Unemployment Rate in the United States in March 2016? Answer with only the number - nothing else in the response.', target='5.0', series_id='UNRATE', series_name='Unemployment Rate', obs_date='2016-03-01', period_start=2015, period_end=2019, tolerance=0.1),
 ObservationEntry(input='What was the Unemployment Rate in the United States in April 2017? Answer with only the number - nothing else in the response.', target='4.4', series_id='UNRATE', series_name='Unemployment Rate', obs_date='2017-04-01', period_start=2015, period_end=2019, tolerance=0.1),
 ObservationEntry(input='What was the Unemployment Rate in the United States in November 2020? Answer with onl

In [76]:
obs_dict = [asdict(obs) for obs in obs_list]

with open("questions.json", "w") as f:
    json.dump(obs_dict, f)

In [ ]:
# Iterate through series, making one API call per year period in a random month
url = "https://api.stlouisfed.org/fred/series/observations"

params = {
    "series_id": None,
    "api_key": api_key,
    "file_type": "json",
    "observation_start": "2015-01-01"
}

for series_id, series_name in SERIES.items():
    print(f"Evaluating series: {series_id} - {series_name}")
    params["series_id"] = series_id
    
    sleep(5)
    response = httpx.get(url, params=params)
    data = json.loads(response.text)
    observations = data["observations"]

    # Iterate through each year period
    series_results = []
    for start, end in YEARS:
        print(series_id, start, end)
        year = random.randint(start, end)
        month = random.choice(MONTHS)

        selected_date = date(year, month, 1)
        print("\t", year, month)

        for obs in observations:
            if obs.get("date") == str(selected_date):
                series_results.append(obs.get("value"))

Evaluating series: UNRATE - Unemployment Rate
UNRATE 2015 2019
	 2018 8
UNRATE 2020 2024
	 2022 8
UNRATE 2025 2025
	 2025 10


In [56]:
pprint(observations)

[{'date': '2015-01-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.7'},
 {'date': '2015-02-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.5'},
 {'date': '2015-03-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.4'},
 {'date': '2015-04-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.4'},
 {'date': '2015-05-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.6'},
 {'date': '2015-06-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.3'},
 {'date': '2015-07-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.2'},
 {'date': '2015-08-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.1'},
 {'date': '2015-09-01',
  'realtime_end': '2026-07-02',
  'realtime_start': '2026-07-02',
  'value': '5.0'},
 {'date': '2015-10-

In [55]:
series_results

['3.8', '3.6', '.']

In [45]:
for start, end in YEARS:
    print(series, start, end)
    year = random.randint(start, end)
    month = random.choice(MONTHS)

    selected_date = date(year, month, 1)
    print(selected_date)

INDPRO 2015 2019
2015-07-01
INDPRO 2020 2024
2024-02-01
INDPRO 2025 2025
2025-07-01


In [ ]:
# Testing random building logic

series_id = "UNRATE"

series_results = []

selected_dates = []
# Loop through each year range
for start, end in YEARS:
    print(f"Evaluating series: {series_id} for {start} - {end}")

    # Pick three random dates in that range, skipping any duplicates
    period_dates = []
    while len(period_dates) < 3:
        year = random.randint(start, end)
        month = random.choice(MONTHS)
        selected = date(year, month, 1)
        selected_str = str(selected)
        print(f"\tSelected date: {selected_str}")

        # Add selected dates to list if not already present
        if selected_str not in period_dates:
            period_dates.append(selected_str)
        else:
            print("\t\tDuplicate - skipping", selected_str)

    selected_dates.extend(period_dates)

series_results = []

for obs in observations:
    obs_date = obs.get("date")
    if obs_date in selected_dates:
        print(f"Date match: {obs_date}")
        series_results.append(obs.get("value"))

series_results